# Hazard Metrics

This notebook creates the Hazard section CSV for the national tool. The initial workflow summarizes JRC river-flood extent and depth for seven return periods; tropical-cyclone metrics can be added as a separate hazard run later.

## 0. Setup

Country settings, administrative level, and source paths come from `config/countries/KEN.toml`. Shared calculations live in `src/national_tool_metrics/sections/hazard.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import validate_section_output, write_section_output
import national_tool_metrics.sections.hazard as hazard_section

importlib.reload(hazard_section)
from national_tool_metrics.sections.hazard import (
    RIVER_FLOOD_MODEL_RUN,
    RIVER_FLOOD_RETURN_PERIODS,
    assemble_hazard_run_metrics,
    build_river_flood_metrics,
)

In [ ]:
config = load_country_config("KEN", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"River-flood return periods: {RIVER_FLOOD_RETURN_PERIODS}")

## 1. River Flood Hazard

Calculate flooded area, administrative-area share, area-weighted mean depth, and area-weighted 90th-percentile depth for RP10, RP20, RP50, RP75, RP100, RP200, and RP500. Raster values are treated as metres of depth, and no-data cells are treated as dry.

In [ ]:
river_flood_metrics = build_river_flood_metrics(config, admin_regions)
river_flood_metrics.head()

## 2. Review Metric Families

Review the flood-extent and flood-depth metric families before assembling the standardized output.

In [ ]:
extent_columns = [
    column for column in river_flood_metrics.columns
    if column == "adm_id" or column.startswith("flooded_area_")
]
depth_columns = [
    column for column in river_flood_metrics.columns
    if column == "adm_id" or column.startswith("flood_depth_")
]

display(river_flood_metrics[extent_columns].head())
display(river_flood_metrics[depth_columns].head())

## 3. Assemble and Validate

Create one baseline JRC river-flood row per administrative region. Return periods remain encoded in metric column names so future tropical-cyclone runs can use additional rows without changing the output grain.

In [ ]:
hazard_metrics = assemble_hazard_run_metrics(
    config,
    admin_regions,
    [river_flood_metrics],
    hazard="river_flood",
    scenario="baseline",
    model_run=RIVER_FLOOD_MODEL_RUN,
)
validate_section_output(hazard_metrics, "hazard")

print(f"Rows: {len(hazard_metrics):,}")
print(f"Metric columns: {len(hazard_metrics.columns) - 9:,}")
hazard_metrics.head()

## 4. Export

Write the canonical Hazard CSV after reviewing the metrics above.

In [ ]:
output_path = write_section_output(hazard_metrics, config, "hazard")
print(f"Exported Hazard metrics to: {output_path}")